# 04 1D CNN on raw ΔQ(V) curves

Instead of hand-made summary statistics, a small 1D CNN (`src/cnn.py`) reads the curves directly. The input has five channels, Q_k(V) − Q_10(V) for k = 20, 40, 60, 80, 100, downsampled to 250 voltage points. The target is standardised log10(cycle life).

8 of the 41 training cells are held out for early stopping. Five seeds are trained and averaged, and the single-seed spread is reported too, since one lucky seed would be misleading.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")

import joblib
import numpy as np
import torch
from src.config import MODELS_DIR
from src.data import load_cache
from src.splits import SPLITS
from src.features import build_feature_table
from src.cnn import train_cnn, predict_cnn, save_cnn
from src.evaluate import score

cells = load_cache()
tables = {name: build_feature_table(cells, ids) for name, ids in SPLITS.items()}
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

device: cuda


## Train five seeds

In [2]:
train_ids = list(tables["train"].index)
rng = np.random.default_rng(0)
val_ids = list(rng.choice(train_ids, size=8, replace=False))
fit_ids = [c for c in train_ids if c not in val_ids]

def curves(ids):
    return [cells[i]["qdlin"] for i in ids]

def lives(ids):
    return tables["train"].loc[ids, "cycle_life"].values

cnns = [train_cnn(curves(fit_ids), lives(fit_ids), curves(val_ids), lives(val_ids), seed=s) for s in range(5)]

## Evaluate

In [3]:
results, spread = {}, {}
for split in ["primary_test", "secondary_test"]:
    ids, truth = list(tables[split].index), tables[split]["cycle_life"]
    per_seed = np.array([predict_cnn(m, curves(ids)) for m in cnns])
    spread[split] = [score(truth, p)["RMSE (cycles)"] for p in per_seed]
    print(f"{split:15s} single seeds RMSE: {spread[split]} | ensemble: {score(truth, per_seed.mean(0))}")

results["1D CNN (5-seed ensemble)"] = {
    s: (t["cycle_life"], np.mean([predict_cnn(m, curves(list(t.index))) for m in cnns], axis=0))
    for s, t in tables.items()
}

for s, m in enumerate(cnns):
    save_cnn(m, MODELS_DIR / f"cnn_seed{s}.pt")
joblib.dump(results, MODELS_DIR / "results_cnn.joblib")
joblib.dump(spread, MODELS_DIR / "cnn_seed_spread.joblib");

primary_test    single seeds RMSE: [326.2, 304.7, 291.1, 298.8, 393.6] | ensemble: {'RMSE (cycles)': 318.5, 'MAPE (%)': 15.0}
secondary_test  single seeds RMSE: [372.0, 366.7, 299.7, 344.9, 507.9] | ensemble: {'RMSE (cycles)': 373.3, 'MAPE (%)': 21.0}


The CNN is worse than the elastic net on both test sets and changes a lot from seed to seed. With only 33 cells to fit on, this isn't surprising. I kept the architecture small and didn't tune it further, since the only way to "improve" it from here would be to look at the test sets.